# Silver Quality Verification Compared to Bronze

This notebook checks the main Bronze and Silver outputs with **10 compact quality topics**:
1. file availability
2. row and column counts
3. column audit
4. year coverage
5. null values
6. duplicate checks
7. business rule checks
8. rejected row summary
9. country alignment across datasets
10. metadata quality


## Imports

In [7]:
import pandas as pd
from pathlib import Path
from IPython.display import display


## Paths

In [8]:
root_dir = Path.cwd()
if not (root_dir / "data").exists():
    root_dir = root_dir.parent
if not (root_dir / "data").exists():
    raise FileNotFoundError("Could not locate project root containing the data directory.")

paths = {
    "bronze_matches": root_dir / "data" / "bronze" / "bronze_wc_matches.csv",
    "bronze_pop": root_dir / "data" / "bronze" / "bronze_pop.csv",
    "bronze_gdp": root_dir / "data" / "bronze" / "bronze_gdp.csv",
    "silver_worldcup": root_dir / "data" / "silver" / "silver_wc_output.csv",
    "silver_socio": root_dir / "data" / "silver" / "valid_socioeconomics.csv",
    "rejected_socio": root_dir / "data" / "silver" / "rejected" / "rejected_socioeconomics.csv",
}

def load_csv(path: Path):
    return pd.read_csv(path) if path.exists() else None

datasets = {name: load_csv(path) for name, path in paths.items()}

availability = pd.DataFrame(
    [
        {
            "dataset": name,
            "exists": df is not None,
            "path": str(path.relative_to(root_dir)),
        }
        for name, path in paths.items()
        for df in [datasets[name]]
    ]
)

display(availability)


,dataset,exists,path
0,bronze_matches,True,data/bronze/bronze_wc_matches.csv
1,bronze_pop,True,data/bronze/bronze_pop.csv
2,bronze_gdp,True,data/bronze/bronze_gdp.csv
3,silver_worldcup,True,data/silver/silver_wc_output.csv
4,silver_socio,True,data/silver/valid_socioeconomics.csv
5,rejected_socio,True,data/silver/rejected/rejected_socioeconomics.csv


## file availability, row/column counts, and column audit

In [9]:

overview_rows = []
for name, df in datasets.items():
    if df is None:
        overview_rows.append({"dataset": name, "rows": pd.NA, "columns": pd.NA, "distinct_columns": pd.NA})
        continue
    overview_rows.append(
        {
            "dataset": name,
            "rows": len(df),
            "columns": df.shape[1],
            "distinct_columns": df.columns.nunique(),
        }
    )

display(pd.DataFrame(overview_rows))

retention_rows = []
if datasets["bronze_matches"] is not None and datasets["silver_worldcup"] is not None:
    bronze_matches_1998_2018 = datasets["bronze_matches"][datasets["bronze_matches"]["year"].between(1998, 2018)]
    retention_rows.append(
        {
            "pipeline": "worldcup",
            "rows_in_scope": len(bronze_matches_1998_2018),
            "silver_rows": len(datasets["silver_worldcup"]),
            "retention_pct": round(len(datasets["silver_worldcup"]) / len(bronze_matches_1998_2018) * 100, 2) if len(bronze_matches_1998_2018) else pd.NA,
        }
    )

if datasets["silver_socio"] is not None and datasets["rejected_socio"] is not None:
    total_reviewed = len(datasets["silver_socio"]) + len(datasets["rejected_socio"])
    retention_rows.append(
        {
            "pipeline": "socioeconomics",
            "rows_in_scope": total_reviewed,
            "silver_rows": len(datasets["silver_socio"]),
            "retention_pct": round(len(datasets["silver_socio"]) / total_reviewed * 100, 2) if total_reviewed else pd.NA,
        }
    )

if retention_rows:
    display(pd.DataFrame(retention_rows))

for name, df in datasets.items():
    if df is None:
        continue
    print(f"\n{name} columns ({len(df.columns)}):")
    print(df.columns.tolist())


,dataset,rows,columns,distinct_columns
0,bronze_matches,900,18,18
1,bronze_pop,266,74,74
2,bronze_gdp,266,74,74
3,silver_worldcup,384,14,14
4,silver_socio,5158,7,7
5,rejected_socio,176,6,6


,pipeline,rows_in_scope,silver_rows,retention_pct
0,worldcup,384,384,100.0
1,socioeconomics,5334,5158,96.7



bronze_matches columns (18):
['year', 'country', 'city', 'stage', 'home_team', 'away_team', 'home_score', 'away_score', 'outcome', 'win_conditions', 'winning_team', 'losing_team', 'date', 'month', 'dayofweek', 'source_name', 'load_timestamp', 'run_id']

bronze_pop columns (74):
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70', 'source_name', 'load_timestamp', 'run_id']

bronze_gdp columns (74):
['Country Name', 'Country Code', 'Indic

## year coverage, null values, and duplicate checks

In [10]:
def extract_year_coverage(df: pd.DataFrame, dataset_name: str):
    if df is None:
        return {"dataset": dataset_name, "year_min": pd.NA, "year_max": pd.NA, "distinct_years": pd.NA}
    if "year" in df.columns:
        year_series = pd.to_numeric(df["year"], errors="coerce")
        return {
            "dataset": dataset_name,
            "year_min": int(year_series.min()) if year_series.notna().any() else pd.NA,
            "year_max": int(year_series.max()) if year_series.notna().any() else pd.NA,
            "distinct_years": int(year_series.nunique()) if year_series.notna().any() else pd.NA,
        }
    year_columns = [int(col) for col in df.columns if str(col).isdigit()]
    return {
        "dataset": dataset_name,
        "year_min": min(year_columns) if year_columns else pd.NA,
        "year_max": max(year_columns) if year_columns else pd.NA,
        "distinct_years": len(year_columns) if year_columns else pd.NA,
    }

year_summary = pd.DataFrame([extract_year_coverage(df, name) for name, df in datasets.items()])
display(year_summary)

null_rows = []
for name, df in datasets.items():
    if df is None:
        continue
    null_counts = df.isna().sum()
    null_rows.append(
        {
            "dataset": name,
            "columns_with_nulls": int((null_counts > 0).sum()),
            "total_null_cells": int(null_counts.sum()),
            "top_null_columns": ", ".join(null_counts[null_counts > 0].sort_values(ascending=False).head(5).index.tolist()) or "none",
        }
    )

display(pd.DataFrame(null_rows))

duplicate_rows = []
key_map = {
    "bronze_matches": ["year", "stage", "home_team", "away_team"],
    "silver_worldcup": ["year", "stage", "home_team", "away_team"],
    "silver_socio": ["country_code", "year"],
    "rejected_socio": ["country_code", "year"],
}
for name, df in datasets.items():
    if df is None:
        continue
    keys = key_map.get(name)
    duplicate_rows.append(
        {
            "dataset": name,
            "full_row_duplicates": int(df.duplicated().sum()),
            "business_key_duplicates": int(df.duplicated(subset=keys).sum()) if keys else pd.NA,
            "business_key": ", ".join(keys) if keys else "n/a",
        }
    )

display(pd.DataFrame(duplicate_rows))


,dataset,year_min,year_max,distinct_years
0,bronze_matches,1930,2018,21
1,bronze_pop,1960,2025,66
2,bronze_gdp,1960,2025,66
3,silver_worldcup,1998,2018,6
4,silver_socio,1998,2018,21
5,rejected_socio,1998,2018,21


,dataset,columns_with_nulls,total_null_cells,top_null_columns
0,bronze_matches,3,1176,"win_conditions, winning_team, losing_team"
1,bronze_pop,67,627,"Unnamed: 70, 2025, 1977, 1961, 1989"
2,bronze_gdp,67,3261,"Unnamed: 70, 2025, 1960, 1961, 1963"
3,silver_worldcup,2,146,"winning_team, losing_team"
4,silver_socio,0,0,none
5,rejected_socio,2,197,"gdp_per_capita_usd, population"


,dataset,full_row_duplicates,business_key_duplicates,business_key
0,bronze_matches,0,9,"year, stage, home_team, away_team"
1,bronze_pop,0,<NA>,n/a
2,bronze_gdp,0,<NA>,n/a
3,silver_worldcup,0,104,"year, stage, home_team, away_team"
4,silver_socio,0,0,"country_code, year"
5,rejected_socio,0,0,"country_code, year"


## business rule checks and rejected row summary

In [11]:
rule_rows = []

silver_worldcup = datasets["silver_worldcup"]
if silver_worldcup is not None:
    draw_without_winner = silver_worldcup["winning_team"].isna() & silver_worldcup["losing_team"].isna()
    rule_rows.extend(
        [
            {"dataset": "silver_worldcup", "check": "home_score >= 0", "issue_count": int((pd.to_numeric(silver_worldcup["home_score"], errors="coerce") < 0).sum())},
            {"dataset": "silver_worldcup", "check": "away_score >= 0", "issue_count": int((pd.to_numeric(silver_worldcup["away_score"], errors="coerce") < 0).sum())},
            {"dataset": "silver_worldcup", "check": "date_clean parsed", "issue_count": int(silver_worldcup.get("date_clean", pd.Series(dtype="object")).isna().sum()) if "date_clean" in silver_worldcup.columns else pd.NA},
        ]
    )

silver_socio = datasets["silver_socio"]
if silver_socio is not None:
    rule_rows.extend(
        [
            {"dataset": "silver_socio", "check": "population > 0", "issue_count": int((pd.to_numeric(silver_socio["population"], errors="coerce") <= 0).sum())},
            {"dataset": "silver_socio", "check": "gdp_per_capita_usd > 0", "issue_count": int((pd.to_numeric(silver_socio["gdp_per_capita_usd"], errors="coerce") <= 0).sum())},
            {"dataset": "silver_socio", "check": "country_code length = 3", "issue_count": int(silver_socio["country_code"].astype("string").str.len().ne(3).sum())},
            {"dataset": "silver_socio", "check": "year between 1998 and 2018", "issue_count": int((~pd.to_numeric(silver_socio["year"], errors="coerce").between(1998, 2018)).sum())},
        ]
    )

display(pd.DataFrame(rule_rows))

if datasets["rejected_socio"] is not None:
    rejected = datasets["rejected_socio"]
    rejection_breakdown = (
        rejected["review_reason"]
        .value_counts(dropna=False)
        .rename_axis("review_reason")
        .reset_index(name="rows")
    )
    rejection_breakdown["share_pct"] = (rejection_breakdown["rows"] / len(rejected) * 100).round(2)
    display(rejection_breakdown)


,dataset,check,issue_count
0,silver_worldcup,home_score >= 0,0
1,silver_worldcup,away_score >= 0,0
2,silver_worldcup,date_clean parsed,0
3,silver_socio,population > 0,0
4,silver_socio,gdp_per_capita_usd > 0,0
5,silver_socio,country_code length = 3,0
6,silver_socio,year between 1998 and 2018,0


,review_reason,rows,share_pct
0,MISSING_GDP,155,88.07
1,MISSING_POPULATION,21,11.93


## country alignment and metadata quality

In [12]:
alignment_rows = []
if datasets["silver_worldcup"] is not None and datasets["silver_socio"] is not None:
    socio_countries = set(datasets["silver_socio"]["country_name"].dropna().astype(str).str.strip())
    host_countries = set(datasets["silver_worldcup"]["country"].dropna().astype(str).str.strip())
    team_countries = set(
        pd.concat(
            [
                datasets["silver_worldcup"]["home_team"],
                datasets["silver_worldcup"]["away_team"],
                datasets["silver_worldcup"]["winning_team"],
                datasets["silver_worldcup"]["losing_team"],
            ],
            ignore_index=True,
        ).dropna().astype(str).str.strip()
    )

    missing_hosts = sorted(host_countries - socio_countries)
    missing_teams = sorted(team_countries - socio_countries)

    alignment_rows.extend(
        [
            {"comparison": "host countries missing from silver_socio", "issue_count": len(missing_hosts), "examples": ", ".join(missing_hosts[:10]) or "none"},
            {"comparison": "team countries missing from silver_socio", "issue_count": len(missing_teams), "examples": ", ".join(missing_teams[:10]) or "none"},
        ]
    )

display(pd.DataFrame(alignment_rows))
print('Missing teams :', missing_teams)

metadata_rows = []
for name, df in datasets.items():
    if df is None:
        continue
    metadata_rows.append(
        {
            "dataset": name,
            "has_source_name": "source_name" in df.columns,
            "has_load_timestamp": "load_timestamp" in df.columns,
            "load_timestamp_parse_failures": int(pd.to_datetime(df["load_timestamp"], errors="coerce").isna().sum()) if "load_timestamp" in df.columns else pd.NA,
            "has_run_id": "run_id" in df.columns,
            "run_id_nulls": int(df["run_id"].isna().sum()) if "run_id" in df.columns else pd.NA,
        }
    )

display(pd.DataFrame(metadata_rows))


,comparison,issue_count,examples
0,host countries missing from silver_socio,0,none
1,team countries missing from silver_socio,1,North Korea


Missing teams : ['North Korea']


,dataset,has_source_name,has_load_timestamp,load_timestamp_parse_failures,has_run_id,run_id_nulls
0,bronze_matches,True,True,0,True,0
1,bronze_pop,True,True,0,True,0
2,bronze_gdp,True,True,0,True,0
3,silver_worldcup,True,False,<NA>,True,0
4,silver_socio,True,True,0,False,<NA>
5,rejected_socio,False,False,<NA>,False,<NA>
